# 03 · Preparação para a modelagem

**CRISP-DM fase 3 — preparação dos dados** · Meridiano Imobiliária

---

O notebook [`02-ajustes-dados`](02-ajustes-dados.ipynb) entregou a **base curada**: 2.350
imóveis, 83 colunas, nenhuma célula vazia e cada ajuste com motivo registrado. Ela descreve
a carteira em **unidades de negócio** — pés quadrados, dólares, notas de 0 a 5 — e é
justamente isso que a torna legível para a Meridiano e imprópria para um algoritmo baseado
em distância.

Este notebook resolve essa diferença. Ele **não corrige mais nenhum dado**: nada é imputado,
nada é removido por qualidade. O que ele faz é responder, com medida na própria base curada,
a sete perguntas de preparação — e cada resposta é uma técnica escolhida contra
alternativas, não por hábito.

| Bloco | Pergunta | Técnica | O que decide |
|:--|:---|:---|:---|
| 2 | Quais das 83 colunas descrevem o **produto**? | Recorte do espaço de atributos | O papel declarado no dicionário e a evidência de que o bairro é preço disfarçado |
| 3 | Duas colunas dizem a mesma coisa? | Correlação de Spearman entre pares | Regra: \|ρ\| ≥ 0,95 é duplicata |
| 4 | Os indicadores binários acrescentam algo? | Concentração e escore padronizado do minoritário | Regra: binária concentrada acima de 95% vira eixo de anomalia |
| 5 | Alguma cauda domina a distância? | Assimetria e transformação de forma | Limiar de 0,75 — e a transformação precisa **reduzir** a assimetria |
| 6 | Que peso cada variável tem na distância? | Padronização | Só o escalador que iguala o desvio de todas |
| 7 | Como entram as variáveis nominais? | *One-hot* contra Gower | Uma categoria de um único imóvel vira z ≈ 48 |
| 8 | Dezessete variáveis são muitas? | PCA como diagnóstico | A interpretabilidade do segmento para o negócio |

A regra deste notebook é uma só:

> **Nada é transformado sem medida.** Toda técnica aplicada aqui responde a um número medido
> na base curada, e a decisão contrária — *não* transformar — vale o mesmo e fica registrada
> do mesmo jeito, com o número que a sustenta.

## O que este notebook não faz

Três coisas ficam de fora, de propósito:

* **não escolhe *k* nem roda a segmentação para valer.** O K-Means aparece como **ensaio de
  verificação**: é o instrumento que mede o efeito de uma decisão de preparação sobre a
  geometria, não o modelo do projeto;
* **não grava dado padronizado em disco.** O arquivo de saída sai em unidades originais e a
  transformação viaja como *receita* (`models/preprocessador-euclidiano.joblib`), ajustada
  dentro da validação. Guardar matriz escalada é o caminho mais curto para vazamento entre
  etapas;
* **não leva o rótulo junto.** `preco_venda` não entra no arquivo de modelagem: quem
  precisar dele na avaliação a posteriori religa pelo `pid`. Rótulo que não viaja não vaza
  por descuido.

In [ ]:
# Importações
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from joblib import dump
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    MinMaxScaler,
    OneHotEncoder,
    RobustScaler,
    StandardScaler,
)

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src import config
from src.data import dicionario as dic
from src.eda import estatistica as est
from src.utils import io

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

io.configurar_estilo()

# As figuras deste notebook vão para `docs/imagens/preparacao/`; as tabelas, para
# `reports/tables/preparacao/`. A regra do projeto é que nenhuma figura existe sem
# a tabela que a produziu.
FIGURAS = config.DOCS / "imagens" / "preparacao"
FIGURAS.mkdir(parents=True, exist_ok=True)
SECAO = "preparacao"

# Cores da identidade do projeto (docs/assets/css/brand.css).
BARRA = config.AZUL_COBALTO
REFERENCIA = config.AMARELO
DESTAQUE = config.AZUL_CLARO
ALERTA = config.CORAL
NEUTRO = config.CINZA


def salvar(fig, nome, dados=None):
    """Grava a figura em `docs/imagens/preparacao/` e a tabela que a originou."""
    fig.savefig(FIGURAS / f"{nome}.png", bbox_inches="tight")
    plt.close(fig)
    if dados is not None:
        io.salvar_tabela(dados, SECAO, nome, indice=False)


print(f"Projeto em {RAIZ}")

## O registro de decisões

O notebook anterior registrou *ajustes* — o que mudou no dado. Este registra **decisões de
preparação**: o que entra no espaço de modelagem, o que sai, o que é transformado e o que
foi conferido e deixado como está.

A diferença entre os dois registros importa. Lá, o que não estava registrado era bug; aqui,
o que não estiver registrado é uma escolha de modelagem feita sem justificativa — e é
exatamente esse tipo de escolha que faz um resultado de agrupamento ser indefensável seis
meses depois.

| `acao` | Significado |
|:---|:---|
| `selecionado` | A coluna entra no espaço de atributos |
| `reservado` | Fica fora da modelagem por decisão de projeto, e o motivo é declarado |
| `removido` | Era candidata e saiu, com a medida que a eliminou ao lado |
| `transformado` | A variável entra, mas em outra forma |
| `verificado, sem ação` | A técnica foi testada e a decisão foi **não** aplicá-la |

In [ ]:
DECISOES: list[dict] = []


def pt(numero) -> str:
    """Formata o número no padrão brasileiro: ponto como separador de milhar."""
    return f"{numero:,}".replace(",", ".")


def registrar(
    bloco: str,
    decisao: str,
    tecnica: str,
    motivo: str,
    evidencia: str,
    acao: str = "selecionado",
):
    """Anota uma decisão de preparação, com a técnica, o motivo e a medida.

    O campo `evidencia` é o que separa este registro de uma lista de boas intenções:
    guarda o **número medido na base curada** que sustenta a decisão.
    """
    DECISOES.append(
        {
            "n": len(DECISOES) + 1,
            "bloco": bloco,
            "decisao": decisao,
            "tecnica": tecnica,
            "motivo": motivo,
            "evidencia": evidencia,
            "acao": acao,
        }
    )
    print(f"{len(DECISOES):>2}. [{acao:>20}] {decisao} — {evidencia}")

# 1 · A entrada — a base curada, e só ela

> A camada bruta em `data/raw/` não é lida aqui, e `data/processed/imoveis_curados.parquet`
> não é reescrito. Este notebook tem **uma entrada e uma saída**.

A conferência abaixo não é decorativa. Se o notebook 02 rodar de novo com outro limiar de
ausência ou outro fator de discrepância, a base curada muda — e as decisões que este
notebook toma sobre concentração, assimetria e escala mudam junto. Confrontar o formato
lido com o `manifesto_curadoria.json` é o que garante que as duas fases estão falando da
**mesma carteira**.

In [ ]:
curada = pd.read_parquet(config.BASE_CURADA)
manifesto_curadoria = io.ler_json(config.DATA_PROCESSED / "manifesto_curadoria.json")

LINHAS, COLUNAS = curada.shape

assert int(curada.isna().sum().sum()) == 0, "a base curada não pode ter célula vazia"
assert LINHAS == manifesto_curadoria["linhas_saida"], "linhas divergem do manifesto"
assert COLUNAS == manifesto_curadoria["colunas_saida"], "colunas divergem do manifesto"

entrada = pd.Series(
    {
        "arquivo": config.BASE_CURADA.relative_to(RAIZ).as_posix(),
        "imóveis": pt(LINHAS),
        "colunas": COLUNAS,
        "células vazias": int(curada.isna().sum().sum()),
        "removidos na curadoria": (
            f"{pt(manifesto_curadoria['registros_removidos'])} "
            f"({manifesto_curadoria['pct_registros_removidos']}% da base bruta)"
        ),
        "ajustes registrados": manifesto_curadoria["ajustes"],
        "equivalente ao pipeline": manifesto_curadoria["equivalente_ao_pipeline"],
    }
)
display(entrada.to_frame("entrada deste notebook"))
curada.head(3)

# 2 · Seleção — quais das 83 colunas descrevem o produto

A primeira técnica de preparação não transforma nada: **escolhe**. E é a escolha que mais
afeta o resultado, porque agrupamento não tem rótulo para corrigir uma variável mal
escolhida. Se `preco_venda` entrar no espaço, os grupos saem por faixa de preço e a
"descoberta" é a tautologia de que imóvel caro se parece com imóvel caro.

Três exclusões estruturam o recorte, e nenhuma delas é sobre qualidade do dado:

**1 · O rótulo reservado.** `preco_venda` fica fora de tudo até a fase 5. Junto com ele sai
`preco_por_pe2`, que dele deriva — variável derivada de rótulo é rótulo, e é o vazamento
mais fácil de cometer sem perceber.

**2 · O bairro.** `bairro` explica **57,4% da variação do preço** (razão de correlação η²
medida adiante, sobre 28 bairros) — nenhuma outra nominal chega perto. Localização é o
principal determinante de preço no mercado imobiliário, e incluí-la faria o algoritmo
redescobrir o preço por caminho indireto. Fica reservada para a **validação externa**: se
os segmentos encontrados sem o bairro tiverem composição de bairro distinta, isso é
evidência independente de que a segmentação capturou algo real.

**3 · O que descreve a transação, não o imóvel.** Mês, ano, tipo e condição de venda
descrevem o *negócio*. Dois imóveis idênticos vendidos em meses diferentes não são
produtos diferentes.

Sobra o quarto grupo, o maior: as **colunas descritivas** que ficaram na base curada e não
entram no espaço. Não é descarte silencioso — a maioria já está representada numa
derivada. As cinco áreas de varanda estão em `area_varandas`; os quatro contadores de
banheiro, em `banheiros_total`; `ano_construcao` e `ano_reforma` viraram `idade_imovel` e
`anos_desde_reforma`; `area_terreo` e `area_pavimento_superior` somam `area_habitavel`.
Elas permanecem na base curada porque a leitura de negócio as usa; entrar de novo no
espaço de distância seria contar a mesma medida duas vezes.

In [ ]:
# O papel de cada coluna, declarado no dicionário e conferido aqui contra a base.
# `preco_por_pe2` recebe papel próprio: é derivada do rótulo, e o motivo de sair
# não é o mesmo das demais descritivas.
CANDIDATAS_EUCLIDIANO = dic.NUMERICAS + dic.ORDINAIS
CANDIDATAS_NOMINAIS = dic.NOMINAIS

PAPEIS = {
    "espaço euclidiano (candidata)": CANDIDATAS_EUCLIDIANO,
    "espaço misto (candidata)": CANDIDATAS_NOMINAIS,
    "indicador binário (bloco 4)": dic.BINARIAS,
    "identificador": dic.IDENTIFICADORES,
    "rótulo reservado": [dic.ROTULO_RESERVADO, "preco_por_pe2"],
    "reservada para validação": [dic.RESERVADA_VALIDACAO],
    "atributo da transação": dic.ATRIBUTOS_TRANSACAO,
}

MOTIVOS = {
    "espaço euclidiano (candidata)": "Mede tamanho, idade, contagem ou nota do imóvel",
    "espaço misto (candidata)": "Categoria sem ordem: entra por Gower, nunca como inteiro",
    "indicador binário (bloco 4)": "Flag derivada — decidida pela medida do bloco 4",
    "identificador": "Rastreabilidade; não descreve o produto",
    "rótulo reservado": "Resultado da venda: uso exclusivo na avaliação a posteriori",
    "reservada para validação": "Proxy de preço; reservada para validação externa",
    "atributo da transação": "Descreve o negócio, não o produto",
    "descritiva, fora do espaço": "Já representada numa derivada ou sem uso na distância",
}


def papel_da(coluna: str) -> str:
    """Devolve o papel declarado da coluna no espaço de modelagem."""
    for nome, colunas in PAPEIS.items():
        if coluna in colunas:
            return nome
    return "descritiva, fora do espaço"


papeis = pd.DataFrame(
    {"coluna": curada.columns, "papel": [papel_da(c) for c in curada.columns]}
).assign(motivo=lambda t: t.papel.map(MOTIVOS))

resumo_papeis = (
    papeis.groupby("papel")
    .agg(colunas=("coluna", "size"), exemplos=("coluna", lambda s: ", ".join(s.head(3))))
    .sort_values("colunas", ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
cores = [
    BARRA if "candidata" in nome else (REFERENCIA if "binário" in nome else NEUTRO)
    for nome in resumo_papeis.papel
]
ax.barh(resumo_papeis.papel, resumo_papeis.colunas, color=cores)
ax.bar_label(ax.containers[0], fmt="%d", padding=3, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Colunas")
ax.set_title(f"Papel das {COLUNAS} colunas da base curada — só as azuis são candidatas")
ax.margins(x=0.12)

salvar(fig, "papel-das-colunas", papeis)
io.salvar_tabela(resumo_papeis, SECAO, "resumo-dos-papeis", indice=False)

display(resumo_papeis)
papeis.query("papel == 'descritiva, fora do espaço'").head(12)

## A evidência das reservadas

A afirmação "o bairro é preço disfarçado" precisa de número. A **razão de correlação**
(η²) mede quanto da variação de uma variável numérica é explicada pela categoria: é a
fração da variância do preço que fica *entre* bairros, e não *dentro* de cada um.

O resultado separa o bairro de todas as demais nominais por uma distância que não deixa
dúvida sobre qual delas carregaria o preço para dentro do espaço de atributos.

In [ ]:
reservadas = pd.DataFrame(
    [
        {
            "variavel": coluna,
            "categorias": int(curada[coluna].nunique()),
            "eta2_sobre_preco": round(
                est.razao_correlacao(curada[coluna], curada[dic.ROTULO_RESERVADO]), 3
            ),
            "papel": papel_da(coluna),
        }
        for coluna in [dic.RESERVADA_VALIDACAO, *dic.NOMINAIS]
    ]
).sort_values("eta2_sobre_preco", ascending=False)

fig, ax = plt.subplots(figsize=(11, 4.5))
cores = [ALERTA if v == dic.RESERVADA_VALIDACAO else BARRA for v in reservadas.variavel]
ax.barh([dic.rotulo(v) for v in reservadas.variavel], reservadas.eta2_sobre_preco, color=cores)
ax.bar_label(ax.containers[0], fmt="%.3f", padding=3, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("η² — fração da variação do preço explicada pela categoria")
ax.set_title("O bairro explica o preço; as nominais do espaço, não")
ax.margins(x=0.15)

salvar(fig, "evidencia-das-reservadas", reservadas)

registrar(
    "2 · Seleção",
    f"Reservar `{dic.ROTULO_RESERVADO}` e `preco_por_pe2`",
    "Exclusão do rótulo e de suas derivadas",
    "O preço é o resultado que a segmentação precisa explicar, não a entrada que a "
    "produz. Variável derivada de rótulo é rótulo.",
    "2 colunas fora do espaço",
    acao="reservado",
)
registrar(
    "2 · Seleção",
    f"Reservar `{dic.RESERVADA_VALIDACAO}` para validação externa",
    "Exclusão de proxy do rótulo",
    "Localização é o principal determinante de preço; incluí-la faria o algoritmo "
    "redescobrir o preço por caminho indireto. Fora do espaço, ela vira evidência "
    "independente de que os segmentos são reais.",
    f"η² = {reservadas.set_index('variavel').loc[dic.RESERVADA_VALIDACAO, 'eta2_sobre_preco']}"
    f" sobre o preço, em {int(curada[dic.RESERVADA_VALIDACAO].nunique())} bairros",
    acao="reservado",
)
registrar(
    "2 · Seleção",
    "Deixar fora identificadores, atributos da transação e descritivas redundantes",
    "Recorte do espaço de atributos",
    "Identificador não descreve produto; mês e condição de venda descrevem o negócio; "
    "as descritivas restantes já estão somadas nas derivadas do notebook 02.",
    f"{int((papeis.papel == 'descritiva, fora do espaço').sum())} descritivas, "
    f"{len(dic.ATRIBUTOS_TRANSACAO)} de transação e {len(dic.IDENTIFICADORES)} identificadores",
    acao="reservado",
)
registrar(
    "2 · Seleção",
    f"Selecionar as {len(CANDIDATAS_EUCLIDIANO)} numéricas e ordinais como candidatas",
    "Recorte do espaço de atributos",
    "Medem tamanho, idade, contagem e nota — o que a Meridiano chama de produto.",
    f"{len(dic.NUMERICAS)} numéricas + {len(dic.ORDINAIS)} ordinais",
)

reservadas

# 3 · Redundância — duas colunas dizendo a mesma coisa

Vinte e uma candidatas entram neste bloco: as 17 numéricas e ordinais mais os 4 indicadores
binários. A pergunta é se alguma delas repete outra.

O motivo de perguntar é aritmético. A distância euclidiana **soma** a contribuição de cada
eixo; duas colunas que medem o mesmo conceito somam duas vezes, e aquele conceito passa a
pesar o dobro na definição dos grupos sem que ninguém tenha decidido isso. Não é opinião de
modelagem: é o que a fórmula faz.

A técnica é a **correlação de Spearman**, e não a de Pearson, por duas razões medidas na
base: metade das candidatas é ordinal ou contagem discreta — nota de 0 a 5, número de
lareiras —, onde a hipótese de relação linear entre níveis não se sustenta; e Spearman lê
relação monótona, que é o que a redundância significa aqui ("quando uma sobe, a outra sobe").

## A regra, declarada antes de olhar a matriz

> **\|ρ\| ≥ 0,95 é duplicata; entre 0,70 e 0,95 é correlação forte, e correlação forte
> fica.**

O corte alto é deliberado. Num conjunto de variáveis que descrevem o mesmo imóvel, `comodos`
e `area_habitavel` correlacionam 0,81 — casa maior tem mais cômodos, e essa é uma
característica *do domínio*, não um defeito do dado. Remover uma das duas jogaria fora
informação real: cômodos pequenos e cômodos amplos são produtos diferentes para a
Meridiano, e só as duas colunas juntas distinguem os dois. Existe ferramenta para
descorrelacionar tudo — PCA, branqueamento —, e o bloco 8 explica por que ela não é
usada aqui.

Acima de 0,95 a leitura muda: não são duas medidas do mesmo conceito, é a **mesma medida
duas vezes**.

In [ ]:
CANDIDATAS = CANDIDATAS_EUCLIDIANO + dic.BINARIAS

LIMIAR_DUPLICATA = 0.95
LIMIAR_FORTE = 0.70

correlacao = curada[CANDIDATAS].corr(method="spearman")
io.salvar_tabela(
    correlacao.reset_index(names="variavel"), SECAO, "correlacao-spearman", indice=False
)

mascara = np.triu(np.ones_like(correlacao, dtype=bool))
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    correlacao,
    mask=mascara,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.4,
    cbar_kws={"shrink": 0.7, "label": "ρ de Spearman"},
    ax=ax,
)
ax.set_title(f"Correlação de Spearman entre as {len(CANDIDATAS)} candidatas")
salvar(fig, "correlacao-entre-candidatas")

pares = est.pares_redundantes(correlacao, limiar=LIMIAR_FORTE)
pares["leitura"] = np.where(
    pares.correlacao.abs() >= LIMIAR_DUPLICATA,
    "duplicata — remover",
    "forte, esperada pelo domínio — manter",
)
io.salvar_tabela(pares, SECAO, "pares-correlacionados", indice=False)
pares

In [ ]:
duplicatas = pares.query("correlacao.abs() >= @LIMIAR_DUPLICATA")

# A lista das que saem do espaço, alimentada aqui e no bloco 4.
REMOVIDAS_DO_ESPACO: list[str] = list(duplicatas.variavel_b)

for _, par in duplicatas.iterrows():
    registrar(
        "3 · Redundância",
        f"Remover `{par.variavel_b}`, duplicata de `{par.variavel_a}`",
        "Correlação de Spearman entre pares",
        "A distância euclidiana soma a contribuição de cada eixo: duas colunas que medem "
        "a mesma coisa fazem aquele conceito pesar o dobro sem decisão de ninguém.",
        f"ρ = {par.correlacao:.3f} com `{par.variavel_a}`",
        acao="removido",
    )

fortes = pares.query("correlacao.abs() < @LIMIAR_DUPLICATA")
registrar(
    "3 · Redundância",
    f"Manter os {len(fortes)} pares entre {LIMIAR_FORTE:.2f} e {LIMIAR_DUPLICATA:.2f}",
    "Correlação de Spearman entre pares",
    "Casa maior tem mais cômodos e imóvel novo tem porão melhor: é característica do "
    "domínio, não defeito do dado. Remover uma das duas jogaria fora a distinção entre "
    "cômodo amplo e cômodo pequeno, que é produto diferente para a Meridiano.",
    f"maior par mantido: {fortes.correlacao.abs().max():.3f}",
    acao="verificado, sem ação",
)

fortes

# 4 · Os indicadores binários — o que a padronização faz com uma flag rara

O notebook 02 derivou quatro flags (`tem_porao`, `tem_garagem`, `tem_lareira`,
`tem_segundo_pavimento`) com o argumento de que "a área sozinha esconde a informação
binária depois de padronizada". O argumento é razoável e **está errado nesta carteira** —
e a única forma de saber isso é medir, que é o que este bloco faz.

Duas medidas bastam.

## 4.1 · A concentração, e o que a padronização faz com ela

Padronizar subtrai a média e divide pelo desvio. Numa binária com proporção *p*, o desvio é
√(p(1−p)) — e quanto mais rara a minoria, menor o desvio e maior o escore de quem está
nela. `tem_porao` vale 1 em 97,3% da carteira: os **63 imóveis sem porão** vão para
**z = −6,02**, e contribuem sozinhos com **36,3** para a distância ao quadrado, quando uma
variável qualquer contribui em torno de 1.

O efeito não é sutil, é dominante. Um algoritmo que minimiza soma de distâncias ao quadrado
vai isolar esses 63 imóveis — não porque eles formem um segmento de negócio, mas porque a
padronização os empurrou para 6 desvios de todo o resto num eixo que vale 0 ou 1.

## 4.2 · A informação já está no espaço

`tem_garagem` tem **V de Cramér = 0,999** com `tipo_garagem`, que já entra pelo espaço
misto; `tem_segundo_pavimento` tem **0,963** com `estilo`, idem; `tem_lareira` correlaciona
**0,970** com `lareiras`, e por isso já saiu no bloco 3. Nenhuma das quatro traz informação
que o espaço não tenha.

## O ensaio que fecha a questão

A tabela e o gráfico acima descrevem geometria. O ensaio abaixo mostra a consequência: o
mesmo K-Means, o mesmo *k* = 4, com e sem as quatro binárias.

Com elas, **um dos quatro grupos é exatamente o conjunto dos 63 imóveis sem porão** — 100%
de pureza, 2,7% da carteira. O Canvas do Problema financia no máximo cinco linhas de
produto; uma delas teria sido gasta para dizer "estes imóveis não têm porão", o que a
Meridiano consulta com um filtro de planilha.

**Decisão: as quatro binárias saem do espaço euclidiano.** O espaço fica com as 17
numéricas e ordinais — exatamente o `ESPACO_NUMERICO` declarado no dicionário.

In [ ]:
LIMIAR_CONCENTRACAO_BINARIA = 0.95

FONTE_DA_FLAG = {
    "tem_porao": "area_porao",
    "tem_garagem": "area_garagem",
    "tem_lareira": "lareiras",
    "tem_segundo_pavimento": "area_pavimento_superior",
}
NOMINAL_EQUIVALENTE = {
    "tem_porao": "qualidade_porao",
    "tem_garagem": "tipo_garagem",
    "tem_lareira": "qualidade_lareira",
    "tem_segundo_pavimento": "estilo",
}

linhas = []
for flag, fonte in FONTE_DA_FLAG.items():
    proporcao = float(curada[flag].mean())
    desvio = float(curada[flag].std())
    minoria = int(min(curada[flag].sum(), len(curada) - curada[flag].sum()))
    z_minoria = (0 - proporcao) / desvio if proporcao > 0.5 else (1 - proporcao) / desvio
    equivalente = NOMINAL_EQUIVALENTE[flag]
    linhas.append(
        {
            "flag": flag,
            "pct_em_1": round(100 * proporcao, 1),
            "minoria": minoria,
            "z_da_minoria": round(z_minoria, 2),
            "contribuicao_para_d2": round(z_minoria**2, 1),
            "correlacao_com_a_fonte": round(
                float(curada[flag].corr(curada[fonte], method="spearman")), 3
            ),
            "cramer_com": equivalente,
            "cramer": round(est.v_de_cramer(curada[flag], curada[equivalente]), 3),
        }
    )

binarias = pd.DataFrame(linhas).sort_values("contribuicao_para_d2", ascending=False)

fig, eixos = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

cores = [ALERTA if p >= 100 * LIMIAR_CONCENTRACAO_BINARIA else BARRA for p in binarias.pct_em_1]
eixos[0].barh([dic.rotulo(f) for f in binarias.flag], binarias.pct_em_1, color=cores)
eixos[0].bar_label(eixos[0].containers[0], fmt="%.1f%%", padding=3, fontsize=9)
eixos[0].axvline(100 * LIMIAR_CONCENTRACAO_BINARIA, color=REFERENCIA, ls="dashed", lw=2)
eixos[0].invert_yaxis()
eixos[0].set_xlabel("% da carteira com a flag em 1")
eixos[0].set_title(f"Concentração — limiar em {100 * LIMIAR_CONCENTRACAO_BINARIA:.0f}%")
eixos[0].margins(x=0.16)

eixos[1].barh([dic.rotulo(f) for f in binarias.flag], binarias.contribuicao_para_d2, color=cores)
eixos[1].bar_label(eixos[1].containers[0], fmt="%.1f", padding=3, fontsize=9)
eixos[1].axvline(1, color=REFERENCIA, ls="dashed", lw=2)
eixos[1].invert_yaxis()
eixos[1].set_xlabel("Contribuição da minoria para a distância ao quadrado")
eixos[1].set_title("Uma variável comum contribui com ~1 (linha tracejada)")
eixos[1].margins(x=0.16)

salvar(fig, "binarias-sob-padronizacao", binarias)
binarias

In [ ]:
# Ensaio de verificação — não é a modelagem do projeto, é o instrumento que mede
# o efeito da decisão. Mesmo k, mesma semente, mesmo escalador: muda só o espaço.
def ensaio(colunas, k=4):
    """Roda um K-Means padronizado sobre as colunas e devolve rótulos e silhueta."""
    dados = StandardScaler().fit_transform(curada[colunas].astype(float))
    rotulos = KMeans(n_clusters=k, n_init=20, random_state=config.SEMENTE).fit_predict(dados)
    return rotulos, float(silhouette_score(dados, rotulos))


com_binarias, silhueta_com = ensaio(CANDIDATAS_EUCLIDIANO + dic.BINARIAS)
sem_binarias, silhueta_sem = ensaio(CANDIDATAS_EUCLIDIANO)

tamanhos = np.bincount(com_binarias)
menor = int(tamanhos.argmin())
sem_porao_no_menor = int((curada.loc[com_binarias == menor, "tem_porao"] == 0).sum())

COM_BINARIAS = len(CANDIDATAS_EUCLIDIANO) + len(dic.BINARIAS)
menor_tamanho = int(tamanhos.min())

comparacao = pd.DataFrame(
    [
        {
            "espaco": f"{COM_BINARIAS} variáveis (com as binárias)",
            "silhueta": round(silhueta_com, 4),
            "menor_grupo": int(tamanhos.min()),
            "pct_da_carteira": round(100 * tamanhos.min() / len(curada), 1),
            "pureza_do_menor": f"{sem_porao_no_menor} dos {menor_tamanho} sem porão",
        },
        {
            "espaco": f"{len(CANDIDATAS_EUCLIDIANO)} variáveis (sem as binárias)",
            "silhueta": round(silhueta_sem, 4),
            "menor_grupo": int(np.bincount(sem_binarias).min()),
            "pct_da_carteira": round(100 * np.bincount(sem_binarias).min() / len(curada), 1),
            "pureza_do_menor": "—",
        },
    ]
)
io.salvar_tabela(comparacao, SECAO, "ensaio-das-binarias", indice=False)

display(pd.crosstab(com_binarias, curada.tem_porao).rename_axis(index="grupo", columns="tem_porao"))

for _, flag in binarias.iterrows():
    if flag.flag == "tem_lareira":
        continue  # já removida no bloco 3, por duplicidade com `lareiras`
    concentrada = flag.pct_em_1 >= 100 * LIMIAR_CONCENTRACAO_BINARIA
    registrar(
        "4 · Binárias",
        f"Remover `{flag.flag}` do espaço euclidiano",
        "Concentração e escore padronizado do minoritário",
        (
            "A padronização de uma binária rara joga a minoria a vários desvios de todo o "
            "resto e cria um eixo de anomalia: o algoritmo isola esses imóveis em vez de "
            "encontrar segmento."
            if concentrada
            else "A informação já está no espaço pela variável de origem e pela nominal "
            "equivalente; a flag só acrescenta um eixo binário que o K-Means usa para partir "
            "a carteira ao meio."
        ),
        (
            f"{flag.pct_em_1}% em 1, minoria de {pt(int(flag.minoria))} imóveis a "
            f"z = {flag.z_da_minoria}"
            if concentrada
            else f"V de Cramér = {flag.cramer} com `{flag.cramer_com}`"
        ),
        acao="removido",
    )

REMOVIDAS_DO_ESPACO += [f for f in FONTE_DA_FLAG if f not in REMOVIDAS_DO_ESPACO]

# O espaço fechado: as 21 candidatas menos o que os blocos 3 e 4 removeram. A
# igualdade com `dic.ESPACO_NUMERICO` não é coincidência — é a leitura do
# dicionário sem as binárias, que é onde a medida foi parar.
ESPACO_EUCLIDIANO = [
    c for c in CANDIDATAS_EUCLIDIANO + dic.BINARIAS if c not in REMOVIDAS_DO_ESPACO
]
assert ESPACO_EUCLIDIANO == dic.ESPACO_NUMERICO, "o espaço final divergiu do dicionário"

registrar(
    "4 · Binárias",
    f"Fixar o espaço euclidiano em {len(ESPACO_EUCLIDIANO)} variáveis",
    "Ensaio de verificação com K-Means (k = 4)",
    "Com as binárias, um dos quatro grupos é exatamente o conjunto dos imóveis sem porão. "
    "O Canvas financia no máximo cinco linhas de produto, e uma delas teria sido gasta "
    "para dizer o que um filtro de planilha responde.",
    f"menor grupo de {int(tamanhos.min())} imóveis, {sem_porao_no_menor} deles sem porão",
)

comparacao

# 5 · Forma — a cauda que domina a distância

Distância euclidiana é soma de diferenças ao quadrado. Numa variável com cauda longa à
direita, os poucos registros da cauda ficam a muitos desvios da massa, e a diferença deles
para todo mundo — elevada ao quadrado — passa a comandar o agrupamento inteiro. É o mesmo
mecanismo do bloco 4, agora numa variável contínua.

A medida é a **assimetria**, com o limiar de 0,75 declarado no projeto
(`dic.LIMIAR_ASSIMETRIA`). Duas observações antes de aplicar:

* **a depuração do notebook 02 já resolveu quase tudo.** `area_terreno` marcava 13,2 na
  base bruta e marca 0,417 aqui — o que parecia forma da distribuição eram lotes de mais de
  200.000 pés², removidos como valores extremos. Transformar agora seria hábito, não
  decisão;
* **as ordinais ficam de fora por natureza.** `qualidade_porao` marca −1,381, e `log1p`
  sobre nota de 0 a 5 destrói o passo constante que dá sentido à escala.

Sobra **uma** contínua acima do limiar: `area_varandas`, com 0,974.

## E aqui a técnica declarada no dicionário falha

`dic.ASSIMETRICAS` manda aplicar `log1p` em `area_varandas`. A medição diz outra coisa:
**16,8% dos imóveis não têm varanda nenhuma**, e `log1p` comprime tanto a cauda direita que
a massa de zeros vira cauda esquerda — a assimetria sai de **0,974 para −1,348**. A
transformação não corrigiu a forma, inverteu.

Isso obriga a escrever a regra que estava implícita:

> **Transformação de forma só se aplica se reduzir \|assimetria\| abaixo do limiar.**
> Transformação que troca uma cauda por outra não corrigiu nada — apenas trocou quem é o
> extremo.

A raiz quadrada passa nesse teste: leva a assimetria a **−0,320**, dentro do limiar, e
mantém a monotonicidade (varanda maior continua sendo varanda maior). É a transformação
escolhida — não por gosto, mas por ser a que a regra aprova.

`dic.ASSIMETRICAS` precisa ser revisto: a declaração lá foi medida sobre uma base anterior
à depuração da carteira.

In [ ]:
TRANSFORMACOES = {
    "nenhuma": lambda s: s,
    "log1p": np.log1p,
    "raiz": np.sqrt,
}

assimetria = (
    curada[ESPACO_EUCLIDIANO]
    .skew()
    .rename("assimetria")
    .to_frame()
    .assign(
        familia=lambda t: np.where(t.index.isin(dic.NUMERICAS), "contínua", "ordinal"),
        acima_do_limiar=lambda t: t.assimetria.abs() >= dic.LIMIAR_ASSIMETRIA,
        pct_zeros=lambda t: [round(100 * float((curada[c] == 0).mean()), 1) for c in t.index],
    )
    .sort_values("assimetria", ascending=False)
    .reset_index(names="variavel")
)

fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(
    data=assimetria,
    x="assimetria",
    y="variavel",
    hue="acima_do_limiar",
    palette={True: ALERTA, False: BARRA},
    dodge=False,
    ax=ax,
)
for grupo in ax.containers:
    ax.bar_label(grupo, fmt="%.2f", fontsize=8, padding=3)
ax.axvline(dic.LIMIAR_ASSIMETRIA, color=REFERENCIA, ls="dashed", lw=2)
ax.axvline(-dic.LIMIAR_ASSIMETRIA, color=REFERENCIA, ls="dashed", lw=2)
ax.set_xlabel(f"Assimetria — limiar do projeto em ±{dic.LIMIAR_ASSIMETRIA}")
ax.set_ylabel("")
ax.set_title("Assimetria das candidatas, na base curada")
ax.legend(title=f"|assimetria| ≥ {dic.LIMIAR_ASSIMETRIA}")
ax.margins(x=0.12)

salvar(fig, "assimetria-das-candidatas", assimetria)

CANDIDATAS_A_TRANSFORMAR = assimetria.query(
    "acima_do_limiar and familia == 'contínua'"
).variavel.tolist()
print(f"Contínuas acima do limiar: {', '.join(CANDIDATAS_A_TRANSFORMAR) or '—'}")
print(f"Declarado em `dic.ASSIMETRICAS`: {', '.join(dic.ASSIMETRICAS)}")
assimetria

In [ ]:
# Qual transformação, se alguma, deixa a variável dentro do limiar.
ensaios = []
for variavel in CANDIDATAS_A_TRANSFORMAR:
    serie = curada[variavel]
    for nome, funcao in TRANSFORMACOES.items():
        transformada = funcao(serie)
        padronizada = (transformada - transformada.mean()) / transformada.std()
        ensaios.append(
            {
                "variavel": variavel,
                "transformacao": nome,
                "assimetria": round(float(transformada.skew()), 3),
                "dentro_do_limiar": abs(float(transformada.skew())) < dic.LIMIAR_ASSIMETRIA,
                "z_maximo": round(float(padronizada.max()), 2),
                "z_minimo": round(float(padronizada.min()), 2),
            }
        )
ensaios = pd.DataFrame(ensaios)
io.salvar_tabela(ensaios, SECAO, "ensaio-de-transformacoes", indice=False)

fig, eixos = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
alvo = CANDIDATAS_A_TRANSFORMAR[0]
for eixo, (nome, funcao) in zip(eixos, TRANSFORMACOES.items()):
    serie = funcao(curada[alvo])
    aprovada = abs(float(serie.skew())) < dic.LIMIAR_ASSIMETRIA
    sns.histplot(serie, bins=40, color=BARRA if aprovada else ALERTA, ax=eixo)
    eixo.set_title(f"{nome} — assimetria {serie.skew():.3f}")
    eixo.set_xlabel(dic.rotulo(alvo))
    eixo.set_ylabel("Imóveis")
fig.suptitle(f"{dic.rotulo(alvo)}: `log1p` inverte a cauda, a raiz corrige", fontsize=13, y=1.06)

salvar(fig, "transformacoes-da-assimetrica", ensaios)

# A transformação escolhida por variável: a primeira que a regra aprova.
TRANSFORMADAS = {}
for variavel in CANDIDATAS_A_TRANSFORMAR:
    aprovadas = ensaios.query(
        "variavel == @variavel and transformacao != 'nenhuma' and dentro_do_limiar"
    )
    if aprovadas.empty:
        registrar(
            "5 · Forma",
            f"Manter `{variavel}` em unidade original",
            "Assimetria e transformação de forma",
            "Nenhuma transformação testada trouxe a assimetria para dentro do limiar.",
            f"assimetria {curada[variavel].skew():.3f}",
            acao="verificado, sem ação",
        )
        continue
    escolhida = aprovadas.sort_values("assimetria", key=abs).iloc[0]
    TRANSFORMADAS[variavel] = escolhida.transformacao
    reprovada = ensaios.query(
        "variavel == @variavel and transformacao != 'nenhuma' and not dentro_do_limiar"
    )
    registrar(
        "5 · Forma",
        f"Aplicar `{escolhida.transformacao}` em `{variavel}`",
        "Assimetria e transformação de forma",
        "Cauda longa domina a distância ao quadrado. A transformação só vale se reduzir a "
        f"assimetria: `{reprovada.iloc[0].transformacao}` a inverteria "
        f"({reprovada.iloc[0].assimetria}), por causa dos "
        f"{assimetria.set_index('variavel').loc[variavel, 'pct_zeros']}% de zeros.",
        f"assimetria {curada[variavel].skew():.3f} → {escolhida.assimetria}",
        acao="transformado",
    )

registrar(
    "5 · Forma",
    "Não transformar as ordinais que passam do limiar",
    "Assimetria e transformação de forma",
    "`log1p` sobre nota de 0 a 5 destrói o passo constante que dá sentido à escala.",
    f"{int(((assimetria.familia == 'ordinal') & assimetria.acima_do_limiar).sum())} ordinais "
    "acima do limiar, nenhuma transformada",
    acao="verificado, sem ação",
)

print(f"Transformações aplicadas: {TRANSFORMADAS or 'nenhuma'}")
ensaios

# 6 · Escala — quem decide o peso de cada variável

Este é o bloco em que a preparação decide, sozinha, qual será o resultado da segmentação —
e é por isso que ele tem experimento em vez de convenção.

O problema está nas unidades. Na base curada, `area_terreno` tem desvio padrão de **3.618,6
pés²** e `qualidade_externa`, de **0,54 ponto** — uma razão de **6.689 para 1**. A distância
euclidiana soma as duas diferenças sem saber que uma está em pés e a outra em nota: uma
diferença corriqueira de 2.000 pés² de terreno entra na conta como 2.000, enquanto a maior
diferença possível de nota de acabamento entra como 3. Sem padronizar, a nota simplesmente
não existe para o algoritmo, e **a variável de maior amplitude numérica é a segmentação**.

## Os quatro tratamentos comparados

| Tratamento | O que faz | O que acontece com o peso |
|:---|:---|:---|
| `sem_escala` | nada | O peso é a unidade de medida — acidente de cadastro |
| `minmax` | leva tudo ao intervalo [0, 1] | Iguala a amplitude, **não** a dispersão: quem tem cauda longa fica com a massa espremida |
| `padrao` | subtrai a média, divide pelo desvio | Iguala a dispersão: toda variável entra com desvio 1 |
| `robusto` | subtrai a mediana, divide pelo IQR | Iguala o miolo, não a dispersão total; protege contra extremos |

A primeira tabela mede exatamente isso: o desvio padrão de cada variável **depois** de cada
tratamento. É a medida direta do peso que cada uma leva para dentro da distância.

O `robusto` merece nota própria. Ele existe para proteger a escala contra valores extremos
— e a carteira que chega aqui já passou pelo corte de 3×IQR do notebook 02. Ele resolve um
problema que esta base não tem mais, e ainda deixa os desvios desiguais.

In [ ]:
ESCALADORES = {
    "sem_escala": None,
    "minmax": MinMaxScaler(),
    "padrao": StandardScaler(),
    "robusto": RobustScaler(),
}


def aplicar_forma(dados: pd.DataFrame) -> pd.DataFrame:
    """Aplica a transformação de forma decidida no bloco 5."""
    saida = dados.astype(float).copy()
    for variavel, nome in TRANSFORMADAS.items():
        saida[variavel] = TRANSFORMACOES[nome](saida[variavel])
    return saida


def escalar(colunas, escalador):
    """Devolve a matriz com a forma corrigida e o escalador aplicado."""
    dados = aplicar_forma(curada[colunas])
    if ESCALADORES[escalador] is None:
        return dados.values
    return type(ESCALADORES[escalador])().fit_transform(dados)


desvios = pd.DataFrame(
    {
        escalador: pd.DataFrame(
            escalar(ESPACO_EUCLIDIANO, escalador), columns=ESPACO_EUCLIDIANO
        ).std()
        for escalador in ESCALADORES
    }
).round(3)

razoes = (
    desvios.max()
    .div(desvios.min())
    .round(1)
    .rename("razao_maior_menor_desvio")
    .to_frame()
    .assign(
        maior_peso=desvios.idxmax(),
        menor_peso=desvios.idxmin(),
    )
    .reset_index(names="escalador")
)

fig, ax = plt.subplots(figsize=(13, 6))
for escalador, cor in zip(ESCALADORES, [NEUTRO, REFERENCIA, config.AZUL_PROFUNDO, DESTAQUE]):
    ax.plot(
        desvios.index,
        desvios[escalador],
        marker="o",
        lw=2,
        color=cor,
        label=escalador,
    )
ax.set_yscale("log")
ax.tick_params(axis="x", rotation=60)
ax.set_ylabel("Desvio padrão depois do tratamento (escala log)")
ax.set_title("Só a padronização deixa todas as variáveis com o mesmo peso na distância")
ax.legend(title="Tratamento")

salvar(fig, "desvio-por-escalador", desvios.reset_index(names="variavel"))
io.salvar_tabela(razoes, SECAO, "razao-de-desvios", indice=False)

display(razoes)
desvios

## O experimento de sensibilidade

A tabela acima mostra o peso; este experimento mostra a consequência. Mesmo espaço, mesmo
K-Means, mesmo *k* = 4 — muda só o escalador. Se as partições discordarem, quem definiu os
segmentos foi o pré-processamento, não a carteira.

A coluna que fecha o argumento é a do **atributo dominante**: a razão entre a variância
entre grupos e a variância dentro dos grupos, medida em unidades originais, revela por qual
variável o algoritmo *de fato* partiu a carteira.

E há uma armadilha para ler junto: `sem_escala` costuma sair com a **maior silhueta**. Não é
bom resultado, é sintoma — um agrupamento praticamente unidimensional é, de fato, bem
separado. A silhueta mede separação geométrica; ela não sabe se o que separou faz sentido de
negócio.

In [ ]:
def atributo_dominante(rotulos) -> tuple[str, float]:
    """Razão entre variância entre grupos e dentro dos grupos, em unidades originais."""
    razoes = {}
    for coluna in ESPACO_EUCLIDIANO:
        serie = curada[coluna].astype(float)
        grupos = serie.groupby(rotulos)
        entre = ((grupos.mean() - serie.mean()) ** 2 * grupos.size()).sum()
        dentro = ((serie - grupos.transform("mean")) ** 2).sum()
        razoes[coluna] = entre / dentro if dentro > 0 else np.nan
    vencedor = max(razoes, key=lambda c: razoes[c])
    return dic.rotulo(vencedor), float(razoes[vencedor])


K_ENSAIO = 4
particoes, linhas = {}, []
for escalador in ESCALADORES:
    dados = escalar(ESPACO_EUCLIDIANO, escalador)
    rotulos = KMeans(n_clusters=K_ENSAIO, n_init=20, random_state=config.SEMENTE).fit_predict(dados)
    particoes[escalador] = rotulos
    tamanhos = np.bincount(rotulos)
    dominante, razao = atributo_dominante(rotulos)
    linhas.append(
        {
            "escalador": escalador,
            "silhueta": round(float(silhouette_score(dados, rotulos)), 4),
            "menor_grupo": int(tamanhos.min()),
            "maior_grupo": int(tamanhos.max()),
            "atributo_dominante": dominante,
            "razao_entre_dentro": round(razao, 2),
        }
    )

sensibilidade = pd.DataFrame(linhas)
ari = pd.DataFrame(
    {
        a: {b: round(float(adjusted_rand_score(particoes[a], particoes[b])), 3) for b in particoes}
        for a in particoes
    }
)

io.salvar_tabela(sensibilidade, SECAO, "sensibilidade-a-escala", indice=False)
io.salvar_tabela(ari.reset_index(names="escalador"), SECAO, "ari-entre-escaladores", indice=False)

fig, eixos = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
cores = [config.AZUL_PROFUNDO if e == "padrao" else NEUTRO for e in sensibilidade.escalador]
eixos[0].bar(sensibilidade.escalador, sensibilidade.razao_entre_dentro, color=cores)
eixos[0].bar_label(
    eixos[0].containers[0],
    labels=[
        f"{r:.2f}\n{a}"
        for r, a in zip(sensibilidade.razao_entre_dentro, sensibilidade.atributo_dominante)
    ],
    padding=3,
    fontsize=8,
)
eixos[0].set_ylabel("Razão entre/dentro do atributo dominante")
eixos[0].set_title("Quanto uma única variável domina a partição — menor é melhor")
eixos[0].margins(y=0.25)

sns.heatmap(ari, annot=True, fmt=".3f", cmap="Blues", vmin=0, vmax=1, square=True, ax=eixos[1])
eixos[1].set_title("ARI entre as partições — quase nada em comum")

salvar(fig, "sensibilidade-a-escala", sensibilidade)

ESCALADOR = "padrao"
por_escalador = razoes.set_index("escalador").razao_maior_menor_desvio
registrar(
    "6 · Escala",
    "Padronizar todas as variáveis do espaço euclidiano (`StandardScaler`)",
    "Comparação de quatro escaladores por desvio resultante e por ARI",
    "É o único tratamento que deixa toda variável com desvio 1, isto é, com o mesmo peso "
    "na distância. Sem escala, o peso é a unidade de cadastro; `minmax` e `robusto` "
    "igualam amplitude e miolo, não dispersão. O `robusto` ainda protege contra extremos "
    "que o corte de 3×IQR já removeu.",
    f"desvios iguais (razão {por_escalador.loc['padrao']}:1) contra "
    f"{por_escalador.loc['sem_escala']}:1 sem escala",
    acao="transformado",
)

display(sensibilidade)
ari

# 7 · As nominais — *one-hot* ou Gower

Cinco variáveis nominais descrevem o imóvel: zoneamento, tipo de construção, estilo, tipo de
garagem e ar central. Elas não têm ordem — `RL` não é maior que `RM` —, então codificá-las
como inteiro seria inventar uma ordem e uma distância que não existem.

Restam dois caminhos, e este bloco escolhe entre eles com medida.

## O que o *one-hot* faz nesta carteira

Codificar as cinco nominais gera **27 colunas binárias**. Duas consequências, ambas
medidas abaixo:

* **diluição.** O espaço passa de 17 para 44 colunas, e as 27 novas respondem por cerca de
  10% da variância total. As variáveis que descrevem tamanho, idade e qualidade — as que o
  negócio usa para definir linha de produto — passam a dividir o espaço com indicadores de
  categoria;
* **a categoria rara.** Oito das 27 categorias aparecem em menos de 1% da carteira, e a mais
  rara aparece em **um único imóvel**. Padronizada, essa coluna joga aquele imóvel para
  **z ≈ 48** — é o problema do bloco 4 outra vez, agora multiplicado por oito. O ARI entre a
  partição do espaço de 17 variáveis e a do espaço com *one-hot* padronizado cai para perto
  de 0,26: são segmentações diferentes, e a diferença foi produzida por categorias que
  descrevem um punhado de imóveis.

## A decisão

**As nominais não entram no espaço euclidiano** — e também não são descartadas. Elas seguem
no arquivo de saída, em unidade original, para o **espaço misto** da fase 4, onde a
dissimilaridade de **Gower** (`src/model/distancias.py`) trata cada tipo pelo que ele é:
diferença normalizada nas numéricas, igualdade nas nominais. Gower não precisa de *one-hot*,
não infla a dimensão e não transforma categoria rara em ponto distante — cada nominal pesa
exatamente uma variável, e não tantas quantas forem suas categorias.

É a mesma informação, avaliada por duas geometrias. A fase 4 compara as duas.

In [ ]:
codificador = OneHotEncoder(drop="if_binary", sparse_output=False).fit(curada[dic.NOMINAIS])
indicadores = codificador.transform(curada[dic.NOMINAIS])
nomes_indicadores = list(codificador.get_feature_names_out())

frequencias = (
    pd.DataFrame(
        {
            "indicador": nomes_indicadores,
            "imoveis": indicadores.sum(axis=0).astype(int),
            "pct_da_carteira": (100 * indicadores.mean(axis=0)).round(2),
        }
    )
    .assign(
        z_do_minoritario=lambda t: [
            round(float((1 - indicadores[:, i].mean()) / indicadores[:, i].std()), 1)
            for i in range(indicadores.shape[1])
        ]
    )
    .sort_values("imoveis")
)
io.salvar_tabela(frequencias, SECAO, "indicadores-das-nominais", indice=False)

cardinalidade = pd.DataFrame(
    [
        {
            "nominal": coluna,
            "categorias": int(curada[coluna].nunique()),
            "categoria_dominante": curada[coluna].value_counts(normalize=True).index[0],
            "pct_dominante": round(100 * curada[coluna].value_counts(normalize=True).iloc[0], 1),
            "colunas_no_one_hot": int(sum(n.startswith(f"{coluna}_") for n in nomes_indicadores)),
        }
        for coluna in dic.NOMINAIS
    ]
)
io.salvar_tabela(cardinalidade, SECAO, "cardinalidade-das-nominais", indice=False)

# O custo do one-hot, medido: dimensão, variância diluída e partição resultante.
espaco_17 = escalar(ESPACO_EUCLIDIANO, ESCALADOR)
espaco_misto = np.hstack([espaco_17, StandardScaler().fit_transform(indicadores)])

rotulos_17 = KMeans(K_ENSAIO, n_init=20, random_state=config.SEMENTE).fit_predict(espaco_17)
rotulos_oh = KMeans(K_ENSAIO, n_init=20, random_state=config.SEMENTE).fit_predict(espaco_misto)

variancia_bruta = np.hstack([espaco_17, indicadores]).var(axis=0)
custo = pd.DataFrame(
    [
        {
            "espaco": f"{espaco_17.shape[1]} variáveis (numéricas e ordinais)",
            "colunas": espaco_17.shape[1],
            "categorias_abaixo_de_1pct": 0,
            "maior_z_de_categoria_rara": "—",
            "ari_contra_o_espaco_de_17": 1.0,
        },
        {
            "espaco": f"{espaco_misto.shape[1]} colunas (com one-hot padronizado)",
            "colunas": espaco_misto.shape[1],
            "categorias_abaixo_de_1pct": int((frequencias.pct_da_carteira < 1).sum()),
            "maior_z_de_categoria_rara": frequencias.z_do_minoritario.max(),
            "ari_contra_o_espaco_de_17": round(
                float(adjusted_rand_score(rotulos_17, rotulos_oh)), 3
            ),
        },
    ]
)
io.salvar_tabela(custo, SECAO, "custo-do-one-hot", indice=False)

fig, ax = plt.subplots(figsize=(14, 6))
cores = [ALERTA if p < 1 else BARRA for p in frequencias.pct_da_carteira]
ax.bar(frequencias.indicador, frequencias.pct_da_carteira, color=cores)
ax.axhline(1, color=REFERENCIA, ls="dashed", lw=2)
ax.set_yscale("log")
ax.tick_params(axis="x", rotation=90, labelsize=7)
ax.set_ylabel("% da carteira (escala log)")
ax.set_title(
    f"As {len(nomes_indicadores)} colunas que o one-hot cria — "
    f"{int((frequencias.pct_da_carteira < 1).sum())} abaixo de 1% da carteira"
)

salvar(fig, "custo-do-one-hot", frequencias)

registrar(
    "7 · Nominais",
    f"Deixar as {len(dic.NOMINAIS)} nominais fora do espaço euclidiano",
    "Comparação entre one-hot e dissimilaridade de Gower",
    "O one-hot leva o espaço de 17 para 44 colunas e transforma categoria de um único "
    "imóvel em ponto a dezenas de desvios do resto. Gower trata cada nominal como uma "
    "variável, sem inflar a dimensão.",
    f"{int((frequencias.pct_da_carteira < 1).sum())} categorias abaixo de 1%, a mais rara a "
    f"z = {frequencias.z_do_minoritario.max()}; ARI {custo.iloc[1].ari_contra_o_espaco_de_17} "
    "contra o espaço de 17",
    acao="reservado",
)
registrar(
    "7 · Nominais",
    "Levar as nominais para o arquivo de saída, em unidade original",
    "Reserva para o espaço misto (Gower)",
    "A informação categórica não é descartada: ela é avaliada por outra geometria na fase "
    "4, e as duas leituras são comparadas.",
    f"{len(dic.NOMINAIS)} colunas no arquivo, nenhuma codificada",
)

display(cardinalidade)
custo

# 8 · Dimensionalidade — o PCA fica como diagnóstico

Dezessete variáveis correlacionadas entre si não ocupam dezessete dimensões de fato. O PCA
mede quanto do espaço é realmente independente: aqui, **8 componentes explicam 80% da
variância** e 11 explicam 90%, com o primeiro componente sozinho respondendo por cerca de
38% — o eixo de tamanho e qualidade que a EDA já havia identificado.

Isso é diagnóstico útil, e não é motivo para trocar as 17 variáveis pelos 8 componentes.

## Por que a redução não entra na preparação

O produto deste projeto é uma **segmentação que a diretoria vai usar para definir linha de
produto**. Um segmento descrito como "média alta em PC1 e baixa em PC3" não é acionável: não
existe imóvel com PC1, existe imóvel com 2.400 pés², nota 8 de acabamento e 12 anos de
idade. Rodar o agrupamento sobre componentes tornaria a perfilagem da fase 4 — a tabela que
diz o que cada segmento é — uma tradução de segunda mão.

O custo de não reduzir é conhecido e pequeno nesta escala: 17 dimensões com 2.350 registros
ficam longe do regime em que a maldição da dimensionalidade achata as distâncias.

**Decisão: o PCA não entra no espaço de modelagem.** Ele permanece como ferramenta de
**visualização** na fase 4 — projetar os segmentos em dois componentes para desenhá-los é
legítimo, porque a projeção é feita *depois* do agrupamento e não define grupo nenhum.

In [ ]:
pca = PCA().fit(espaco_17)
acumulada = np.cumsum(pca.explained_variance_ratio_)

variancia = pd.DataFrame(
    {
        "componente": np.arange(1, len(acumulada) + 1),
        "variancia_explicada": pca.explained_variance_ratio_.round(4),
        "variancia_acumulada": acumulada.round(4),
    }
)
COMPONENTES_80 = int(np.searchsorted(acumulada, 0.80) + 1)
COMPONENTES_90 = int(np.searchsorted(acumulada, 0.90) + 1)

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.bar(
    variancia.componente, 100 * variancia.variancia_explicada, color=BARRA, label="por componente"
)
ax.plot(
    variancia.componente,
    100 * variancia.variancia_acumulada,
    marker="o",
    color=ALERTA,
    lw=2,
    label="acumulada",
)
ax.axhline(80, color=REFERENCIA, ls="dashed", lw=2)
ax.axvline(COMPONENTES_80, color=REFERENCIA, ls="dotted", lw=2)
ax.set_xticks(variancia.componente)
ax.set_xlabel("Componente")
ax.set_ylabel("% da variância")
ax.set_title(
    f"{COMPONENTES_80} componentes explicam 80% das {espaco_17.shape[1]} variáveis — "
    "diagnóstico, não transformação"
)
ax.legend()

salvar(fig, "diagnostico-de-dimensionalidade", variancia)

registrar(
    "8 · Dimensão",
    "Não aplicar redução de dimensionalidade no espaço de modelagem",
    "PCA como diagnóstico",
    "Segmento descrito por componente não é acionável: a diretoria define linha de produto "
    "em pés quadrados e nota de acabamento, não em PC1. Com 2.350 registros em 17 "
    "dimensões, o custo de não reduzir é pequeno.",
    f"{COMPONENTES_80} componentes para 80% da variância, {COMPONENTES_90} para 90%",
    acao="verificado, sem ação",
)

variancia.head(12)

# 9 · O pré-processador — a receita, não o dado

As decisões dos blocos 5 e 6 são duas operações: corrigir a forma de `area_varandas` e
padronizar tudo. Elas poderiam ser aplicadas agora e o resultado gravado em disco. **Não
serão**, e o motivo é o mesmo que o notebook 02 declarou ao gravar a base curada em pés
quadrados e dólares.

Padronizar exige calcular média e desvio **sobre um conjunto de registros**. Se esse cálculo
roda uma vez sobre a carteira inteira e o resultado é gravado, toda validação posterior —
reamostragem por *bootstrap*, subamostra, imóvel novo chegando pela aplicação — herda
médias que já viram os registros que deveriam ser tratados como desconhecidos. O efeito é
sutil e o sintoma é clássico: estabilidade otimista na validação e um modelo que se comporta
pior em produção do que no notebook.

A alternativa é gravar a **receita**. `models/preprocessador-euclidiano.joblib` guarda o
pré-processador *não ajustado*: as etapas, a ordem e as colunas. Quem for usá-lo na fase 4
ajusta dentro da sua própria validação, e a aplicação de implantação serializa o
pré-processador **junto** com o modelo, para que nenhuma camada reimplemente a padronização
por conta própria.

## A ordem das operações

Primeiro a forma, depois a escala. A ordem não é indiferente: padronizar antes da raiz
produziria valores negativos, e raiz de negativo não existe. Corrigida a forma, a
padronização opera sobre a distribuição que vai de fato para a distância.

In [ ]:
UFUNCS = {"log1p": np.log1p, "raiz": np.sqrt}

# Um ramo por variável transformada, mais um ramo com o resto. `np.sqrt` e
# `np.log1p` são funções de módulo — `joblib` as serializa por referência, o que
# uma função anônima não permitiria.
ramos = [
    (
        f"{nome}_{variavel}",
        Pipeline(
            [
                ("forma", FunctionTransformer(UFUNCS[nome], feature_names_out="one-to-one")),
                ("escala", StandardScaler()),
            ]
        ),
        [variavel],
    )
    for variavel, nome in TRANSFORMADAS.items()
]
SEM_TRANSFORMACAO = [c for c in ESPACO_EUCLIDIANO if c not in TRANSFORMADAS]
ramos.append(("padrao", StandardScaler(), SEM_TRANSFORMACAO))

preprocessador = ColumnTransformer(ramos, verbose_feature_names_out=False)

# O ajuste aqui é conferência, não produção: mostra o que a receita faz. O que vai
# para o disco é o pré-processador não ajustado.
espaco = preprocessador.fit_transform(curada[ESPACO_EUCLIDIANO])
nomes_do_espaco = list(preprocessador.get_feature_names_out())

conferencia_escala = (
    pd.DataFrame(espaco, columns=nomes_do_espaco)
    .agg(["mean", "std", "min", "max"])
    .T.round(3)
    .rename(columns={"mean": "media", "std": "desvio", "min": "z_minimo", "max": "z_maximo"})
    .assign(
        transformacao=lambda t: [TRANSFORMADAS.get(c, "—") for c in t.index],
        unidade_original=lambda t: [dic.DESCRICOES.get(c, "—") for c in t.index],
    )
    .reset_index(names="variavel")
)
io.salvar_tabela(conferencia_escala, SECAO, "conferencia-do-preprocessador", indice=False)

assert np.abs(espaco.mean(axis=0)).max() < 1e-9, "alguma variável não ficou centrada"
assert np.allclose(espaco.std(axis=0), 1.0), "alguma variável não ficou com desvio 1"
print(f"Espaço euclidiano: {espaco.shape[0]} imóveis × {espaco.shape[1]} variáveis")
print("Todas centradas em 0 e com desvio 1 — conferido, não presumido.")
conferencia_escala

# 10 · A base de modelagem — o que vai para a fase 4

O arquivo de saída carrega três blocos de colunas, e nada além deles:

| Bloco | Colunas | Para que serve |
|:---|--:|:---|
| Identificadores | 2 | Religar o resultado à base curada e ao rótulo reservado |
| Espaço euclidiano | 17 | Entra na distância, depois do pré-processador |
| Espaço misto | 5 | Nominais em unidade original, para a dissimilaridade de Gower |

Três conferências antes de gravar, e nenhuma é opcional:

1. **o rótulo não está no arquivo** — nem `preco_venda`, nem `preco_por_pe2`, nem `bairro`.
   É a garantia física contra o vazamento: quem carregar este arquivo na fase 4 não tem como
   usar o preço por descuido, porque ele não está lá;
2. **nenhuma célula vazia e todo o espaço euclidiano numérico** — o pré-processador não
   imputa nada, e não deveria mesmo: imputação é decisão de curadoria, e ela aconteceu no
   notebook 02, com registro;
3. **o número de imóveis é o mesmo da base curada** — este notebook não remove registro
   nenhum. Se a contagem mudar, alguma seleção derrubou linha sem declarar.

In [ ]:
COLUNAS_DA_SAIDA = dic.IDENTIFICADORES + ESPACO_EUCLIDIANO + dic.NOMINAIS
base_modelagem = curada[COLUNAS_DA_SAIDA].copy()

PROIBIDAS = {dic.ROTULO_RESERVADO, "preco_por_pe2", dic.RESERVADA_VALIDACAO}
assert not PROIBIDAS & set(base_modelagem.columns), "rótulo ou reservada vazou para a saída"
assert int(base_modelagem.isna().sum().sum()) == 0, "a base de modelagem não pode ter vazio"
assert len(base_modelagem) == LINHAS, "o notebook não pode remover registro"
assert base_modelagem[ESPACO_EUCLIDIANO].select_dtypes("number").shape[1] == len(
    ESPACO_EUCLIDIANO
), "há variável não numérica no espaço euclidiano"
assert base_modelagem["pid"].is_unique, "o identificador precisa ser único"

esquema = pd.DataFrame(
    {
        "coluna": base_modelagem.columns,
        "papel": [
            "identificador"
            if c in dic.IDENTIFICADORES
            else ("espaço euclidiano" if c in ESPACO_EUCLIDIANO else "espaço misto (Gower)")
            for c in base_modelagem.columns
        ],
        "tipo": [str(base_modelagem[c].dtype) for c in base_modelagem.columns],
        "transformacao": [TRANSFORMADAS.get(c, "—") for c in base_modelagem.columns],
        "descricao": [dic.DESCRICOES.get(c, "—") for c in base_modelagem.columns],
    }
)
io.salvar_tabela(esquema, SECAO, "esquema-da-base-de-modelagem", indice=False)

print(f"{pt(len(base_modelagem))} imóveis × {base_modelagem.shape[1]} colunas")
print(f"Rótulo e reservadas fora do arquivo: {', '.join(sorted(PROIBIDAS))}")
esquema

In [ ]:
config.garantir_diretorios()

ARQUIVO_MODELAGEM = config.DATA_PROCESSED / "base_modelagem.parquet"
ARQUIVO_PREPROCESSADOR = config.MODELS / "preprocessador-euclidiano.joblib"

base_modelagem.to_parquet(ARQUIVO_MODELAGEM, index=False)

# Não ajustado, de propósito: a receita viaja, as médias e desvios não.
dump(ColumnTransformer(ramos, verbose_feature_names_out=False), ARQUIVO_PREPROCESSADOR)

registro = pd.DataFrame(DECISOES)
io.salvar_tabela(registro, SECAO, "registro-de-decisoes", indice=False)

manifesto = {
    "origem": config.BASE_CURADA.relative_to(RAIZ).as_posix(),
    "destino": ARQUIVO_MODELAGEM.relative_to(RAIZ).as_posix(),
    "preprocessador": ARQUIVO_PREPROCESSADOR.relative_to(RAIZ).as_posix(),
    "linhas": int(len(base_modelagem)),
    "colunas": int(base_modelagem.shape[1]),
    "identificadores": dic.IDENTIFICADORES,
    "espaco_euclidiano": ESPACO_EUCLIDIANO,
    "espaco_misto": dic.NOMINAIS,
    "removidas_do_espaco": REMOVIDAS_DO_ESPACO,
    "transformacoes_de_forma": TRANSFORMADAS,
    "escalador": ESCALADOR,
    "reservadas_fora_do_arquivo": sorted(PROIBIDAS),
    "componentes_para_80pct": COMPONENTES_80,
    "reducao_aplicada": False,
    "escala_aplicada_no_arquivo": False,
    "semente": config.SEMENTE,
    "decisoes": int(len(registro)),
    "decisoes_por_acao": registro.acao.value_counts().to_dict(),
}
io.salvar_json(manifesto, config.DATA_PROCESSED / "manifesto_preparacao.json")

print(f"Base de modelagem gravada em {ARQUIVO_MODELAGEM.relative_to(RAIZ)}")
print(f"Pré-processador (não ajustado) em {ARQUIVO_PREPROCESSADOR.relative_to(RAIZ)}")
print(
    f"{pt(len(base_modelagem))} imóveis × {base_modelagem.shape[1]} colunas · "
    f"{len(registro)} decisões"
)
pd.Series(manifesto).to_frame("manifesto da preparação")

In [ ]:
# O funil das colunas: de 83 na base curada às que sobrevivem no espaço de distância.
funil = pd.DataFrame(
    [
        {"etapa": "Base curada", "colunas": COLUNAS},
        {
            "etapa": "Depois de reservar rótulo, bairro, transação e descritivas",
            "colunas": len(ESPACO_EUCLIDIANO) + len(REMOVIDAS_DO_ESPACO) + len(dic.NOMINAIS),
        },
        {
            "etapa": "Depois de remover duplicatas e binárias",
            "colunas": len(ESPACO_EUCLIDIANO) + len(dic.NOMINAIS),
        },
        {"etapa": "Espaço euclidiano (entra na distância)", "colunas": len(ESPACO_EUCLIDIANO)},
    ]
)
funil["saíram"] = funil.colunas.shift(1).sub(funil.colunas).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(funil.etapa, funil.colunas, color=BARRA)
ax.bar_label(
    ax.containers[0],
    labels=[f"{c}" + (f"   −{s}" if s else "") for c, s in zip(funil.colunas, funil["saíram"])],
    padding=4,
    fontsize=9,
)
ax.invert_yaxis()
ax.set_xlabel("Colunas")
ax.set_title(
    f"Funil da preparação — {COLUNAS} colunas curadas, "
    f"{len(ESPACO_EUCLIDIANO)} no espaço de distância"
)
ax.margins(x=0.20)

salvar(fig, "funil-da-preparacao", funil)
funil

In [ ]:
registro

In [ ]:
# O que a fase 4 recebe: identificador, espaço euclidiano em unidades de negócio e
# as nominais intactas para o espaço misto.
base_modelagem.head(10)

# Próximo passo

A base de modelagem está gravada, e é a única entrada da fase 4:

* **`data/processed/base_modelagem.parquet`** — 2.350 imóveis, 24 colunas, em unidades
  originais: 2 identificadores, as 17 variáveis do espaço euclidiano e as 5 nominais do
  espaço misto. Sem preço, sem bairro, sem nada derivado do rótulo;
* **`models/preprocessador-euclidiano.joblib`** — a receita não ajustada: raiz em
  `area_varandas`, padronização em todas. Quem usar ajusta dentro da própria validação;
* **`data/processed/manifesto_preparacao.json`** — o que entrou, o que saiu, qual
  transformação e por quê, no mesmo formato dos manifestos anteriores;
* **`reports/tables/preparacao/registro-de-decisoes.csv`** — o registro completo, que a
  documentação publica.

A fase 4 carrega o arquivo, ajusta o pré-processador dentro da validação e compara as duas
geometrias que este notebook deixou preparadas: o **espaço euclidiano** das 17 variáveis
padronizadas e o **espaço misto** por Gower, que traz as nominais de volta sem *one-hot*.
A escolha de *k* e a comparação entre algoritmos começam ali — nenhuma das duas foi
decidida aqui.

## O que voltar para o código

Duas divergências entre o que este notebook mediu e o que `src/data/dicionario.py` declara
ficaram registradas, e as duas são revisão de dicionário, não improviso de notebook:

* **`dic.ASSIMETRICAS`** manda `log1p` em `area_varandas`, e a medição mostra que `log1p`
  inverte a assimetria (0,974 → −1,348) por causa dos 16,8% de imóveis sem varanda. A raiz
  quadrada é a transformação que a regra do projeto aprova (−0,320);
* **`dic.ESPACO_EUCLIDIANO`** inclui as quatro binárias derivadas, e o bloco 4 mostrou que
  duas delas são quase constantes depois da depuração — a ponto de o K-Means gastar um dos
  quatro grupos com os 63 imóveis sem porão. O espaço deste notebook é exatamente
  `dic.ESPACO_NUMERICO`.

Ambas nasceram do mesmo motivo: as declarações foram medidas sobre uma base anterior à
depuração da carteira. É o comportamento esperado de um projeto em que o dicionário é fonte
única da verdade — quando a medida diverge da declaração, é a declaração que se revisa, com
o número na mão.